# Step 4: Advanced Feature Engineering & Spatial Geolocation
**MSc Data Science Thesis - University of Wolverhampton**

###  Domain & Academic Justification of Spatial Parameters
1. **Earth Radius Constant ($R = 6371.0 \text{ km}$)**: International Union of Geodesy and Geophysics (IUGG) standard mean radius of the Earth used for great-circle kilometer conversions.
2. **São Paulo Hub Coordinates ($-23.5505^\circ \text{S}, -46.6333^\circ \text{W}$)**: Represents Olist's primary central logistics hub and headquarters in São Paulo, Brazil. Because >70% of sellers and orders originate from or transit through São Paulo, calculating distance from the customer to São Paulo serves as an accurate **logistics transit distance proxy**.

###  Haversine Great-Circle Distance Equation:
$$d = 2R \cdot \arcsin\left(\sqrt{\sin^2\left(\frac{\Delta \phi}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\left(\frac{\Delta \lambda}{2}\right)}\right)$$

###  Summary of Engineered Features:
- `haversine_distance_km`: Spatial shipping distance from customer location to São Paulo central hub.
- `reviewer_deviance_score`: $|\text{Customer Avg Rating} - \text{Product/Category Avg Rating}|$.
- `freight_to_price_ratio`: $\frac{\text{Freight Cost}}{\text{Product Price}}$ (logistics cost risk ratio).
- `density_g_cm3`: $\frac{\text{Weight (g)}}{\text{Volume (cm}^3)}$ (package physical bulkiness).
- **Output Checkpoint**: `processed_return_data.csv`.

In [1]:
import pandas as pd
import numpy as np

print("Step 4: Commencing Advanced Feature Engineering on leak-free dataset...")

# ============================================================
# 1. LOAD LEAK-FREE DATASET FROM STEP 3
# ============================================================
df = pd.read_csv('processed_return_data_no_leakage.csv')
print(f"Loaded Step 3 dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

# ============================================================
# 2. HAVERSINE SPATIAL DISTANCE
# São Paulo Hub Coordinates: -23.5505, -46.6333
# ============================================================
print("Calculating Haversine spatial distance metrics...")

EARTH_RADIUS_KM = 6371.0
SAO_PAULO_LAT = -23.5505
SAO_PAULO_LON = -46.6333

def calculate_haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate great-circle distance between two points on Earth.
    Returns distance in kilometers.
    """
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return EARTH_RADIUS_KM * c

# Calculate customer-to-hub distance
df['haversine_distance_km'] = calculate_haversine_distance(
    df['customer_lat'],
    df['customer_lon'],
    SAO_PAULO_LAT,
    SAO_PAULO_LON
)

# Impute missing distances with median distance
df['haversine_distance_km'] = (
    df['haversine_distance_km']
      .fillna(df['haversine_distance_km'].median())
)

# ============================================================
# 3. REVIEWER DEVIANCE SCORE (LEAKAGE-FREE)
# ============================================================
print("Calculating reviewer deviance scores and behavioral features...")

# Historical category order count (past orders only)
df['category_order_count'] = (
    df.groupby('product_category_name_english')
      .cumcount()
)

# Historical category review score sum (excluding current order)
df['cum_cat_review_sum'] = (
    df.groupby('product_category_name_english')['review_score']
      .cumsum()
      - df['review_score']
)

# Historical category average review
df['category_avg_review'] = np.where(
    df['category_order_count'] > 0,
    df['cum_cat_review_sum'] / df['category_order_count'],
    3.0  # neutral baseline
)

# Reviewer deviance score
df['reviewer_deviance_score'] = np.abs(
    df['customer_avg_review'] - df['category_avg_review']
)

# ============================================================
# 4. LOGISTICS & PRICING INTERACTION FEATURES
# ============================================================

# Shipping cost relative to item price
df['freight_to_price_ratio'] = np.where(
    df['price'] > 0,
    df['freight_value'] / df['price'],
    np.nan
)

# Package density (grams per cubic centimeter)
df['density_g_cm3'] = np.where(
    df['product_volume_cm3'] > 0,
    df['product_weight_g'] / df['product_volume_cm3'],
    np.nan
)

# Optional: distance bands for EDA visualization
df['distance_band'] = pd.cut(
    df['haversine_distance_km'],
    bins=[0, 100, 300, 600, 1000, 5000],
    labels=['0-100', '100-300', '300-600', '600-1000', '1000+'],
    include_lowest=True
)

# ============================================================
# 5. FINAL VALIDATION
# ============================================================
print("Running final validation checks...")

# Replace infinite values with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Impute remaining numeric NaNs with column medians
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# ============================================================
# 6. SAVE FINAL ENGINEERED DATASET
# ============================================================
output_file = 'processed_return_data.csv'
df.to_csv(output_file, index=False)

print("=" * 70)
print("✅ STEP 4 COMPLETE: ADVANCED FEATURE ENGINEERING SUCCESSFUL")
print("=" * 70)
print(f"Output File : {output_file}")
print(f"Final Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\nSample of Engineered Features:")
print(df[[
    'haversine_distance_km',
    'reviewer_deviance_score',
    'freight_to_price_ratio',
    'density_g_cm3'
]].head())

print("\nSummary Statistics:")
print(df[[
    'haversine_distance_km',
    'reviewer_deviance_score',
    'freight_to_price_ratio',
    'density_g_cm3'
]].describe())

print("\n✅ Dataset is now ready for Step 5 (EDA and Model Training)!")

Step 4: Commencing Advanced Feature Engineering on leak-free dataset...
Loaded Step 3 dataset: 110,739 rows × 46 columns
Calculating Haversine spatial distance metrics...
Calculating reviewer deviance scores and behavioral features...
Running final validation checks...
✅ STEP 4 COMPLETE: ADVANCED FEATURE ENGINEERING SUCCESSFUL
Output File : processed_return_data.csv
Final Shape : 110,739 rows × 54 columns

Sample of Engineered Features:
   haversine_distance_km  reviewer_deviance_score  freight_to_price_ratio  \
0             781.034688                      0.0                0.261513   
1             353.198715                      0.0                0.062903   
2             353.198715                      0.0                0.062903   
3             353.198715                      0.0                0.062903   
4              11.318427                      0.0                0.093400   

   density_g_cm3  
0       0.560000  
1       0.244141  
2       0.244141  
3       0.244141  
4